# Hindsight: LoRA Fine-Tuning on Colab

This notebook runs the Hindsight LoRA fine-tuning pipeline on a free Colab T4 GPU.

## What This Does

1. Collects trajectories using a real LLM policy (Qwen2.5-1.5B)
2. Attributes failures to find the decisive step
3. Builds training datasets (outcome SFT, salvage SFT, DPO)
4. Trains LoRA adapters
5. Evaluates on held-out seeds
6. Compares results with statistical testing

## Estimated Time

- Collect: ~10-15 minutes (300 runs)
- Attribute: ~5-10 minutes
- Build data: ~2 minutes
- Train SFT: ~20-30 minutes
- Train DPO: ~20-30 minutes
- Evaluate: ~5-10 minutes

**Total**: ~1-1.5 hours

## Cost

- Free (Colab T4 GPU)
- No API keys needed (uses local HuggingFace model)

# Step 1: Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/vedantjain09/hindsight.git
%cd hindsight

# Install dependencies
!pip install -q -e '.[train]'

# Verify GPU is available
!nvidia-smi

# Step 2: Collect Trajectories with Real LLM Policy

This collects 300 trajectories using Qwen2.5-1.5B (a real LLM, not the scripted policy).

In [ ]:
!hindsight collect --policy lm --model Qwen/Qwen2.5-1.5B-Instruct --n 300 --db runs/lm.db

# Step 3: Attribute Failures

Run counterfactual replay to find the decisive step in each failed run.

In [ ]:
!hindsight attribute --policy lm --db runs/lm.db

# Step 4: Build Training Datasets

Build three datasets:
- outcome_sft: pairs from decisive step (SFT)
- salvage_sft: good steps from failed runs (SFT)
- dpo: preference pairs for DPO

In [ ]:
!hindsight build-data --policy lm --db runs/lm.db --out-dir runs/data

# Step 5: Train Outcome SFT LoRA

Train a LoRA adapter on the outcome SFT dataset (max 1500 examples).

In [ ]:
!python scripts/train_sft.py --data runs/data/outcome_sft.jsonl --out runs/lora_outcome --max-examples 1500

# Step 6: Train Salvage SFT LoRA

Train a LoRA adapter on the salvage SFT dataset (max 1500 examples).

In [ ]:
!python scripts/train_sft.py --data runs/data/salvage_sft.jsonl --out runs/lora_salvage --max-examples 1500

# Step 7: Train DPO LoRA

Train a LoRA adapter using DPO on the preference pairs.

In [ ]:
!python scripts/train_dpo.py --data runs/data/dpo.jsonl --out runs/lora_dpo

# Step 8: Evaluate on Held-Out Seeds

Evaluate the salvage SFT adapter on held-out seeds (not used in training).

In [ ]:
!python scripts/eval_policy.py --adapter runs/lora_salvage --out runs/eval_salvage.json

# Step 9: Compare Results

Compare outcome-only vs salvage SFT using paired bootstrap + McNemar's test.

In [ ]:
!hindsight compare runs/eval_outcome.json runs/eval_salvage.json

# Step 10: Save Results

Download the trained LoRA adapters and evaluation results.

In [ ]:
# List the results
!ls -lh runs/

# Download LoRA adapters
from google.colab import files
files.download('runs/lora_outcome')
files.download('runs/lora_salvage')
files.download('runs/lora_dpo')

# Download evaluation results
files.download('runs/eval_outcome.json')
files.download('runs/eval_salvage.json')

# Summary

You've now:
1. ✅ Collected real LLM trajectories
2. ✅ Attributed failures to decisive steps
3. ✅ Built training datasets
4. ✅ Trained 3 LoRA adapters (outcome SFT, salvage SFT, DPO)
5. ✅ Evaluated on held-out seeds
6. ✅ Compared results statistically

## Next Steps

- Download the results using the cell above
- Analyze the statistical comparison
- Check if salvage SFT beats outcome-only
- Upload LoRA adapters to Hugging Face if desired